In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "takala/financial_phrasebank",
    "sentences_75agree",
    trust_remote_code=True,
)

dataset

/Users/anushreegoyal/PycharmProjects/FinancialNewsML/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3453
    })
})

In [ ]:
# Inspect the dataset structure
dataset

# Look at some examples
dataset["train"][0:5]






DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 3453
    })
})

In [3]:
# Convert to pandas
df = dataset["train"].to_pandas()

df.head()

,sentence,label
0,"According to Gran , the company has no plans t...",1
1,With the new production plant the company woul...,2
2,"For the last quarter of 2010 , Componenta 's n...",2
3,"In the third quarter of 2010 , net sales incre...",2
4,Operating profit rose to EUR 13.1 mn from EUR ...,2


# Exploratory Data Analysis

In [ ]:
df.info()

## Label distribution and data quality

In [ ]:
label_names = {0: "negative", 1: "neutral", 2: "positive"}
df["sentiment"] = df["label"].map(label_names)

df.head()

Map numeric targets to sentiment names for analysis and visualization.

In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())
print("\nDuplicate sentences:", df["sentence"].duplicated().sum())
print("\nClass counts:")
print(df["sentiment"].value_counts())

The class distribution is imbalanced, with neutral observations forming the majority. Model evaluation should therefore include class-level precision, recall, and F1 scores.

In [ ]:
import matplotlib.pyplot as plt

class_order = ["negative", "neutral", "positive"]
df["sentiment"].value_counts().reindex(class_order).plot(
    kind="bar",
    color=["#d95f5f", "#7f8c8d", "#55a868"],
)
plt.title("Financial PhraseBank class distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of sentences")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Duplicate handling

In [ ]:
duplicate_rows = df[df["sentence"].duplicated(keep=False)]
duplicate_rows.sort_values("sentence")

Review duplicated sentences and their labels before removal.

In [ ]:
df = df.drop_duplicates(subset="sentence").reset_index(drop=True)

print("Rows after removing duplicates:", len(df))
print("Duplicates remaining:", df["sentence"].duplicated().sum())

Five duplicate sentences had consistent labels and were removed before splitting to prevent train-test leakage. The cleaned dataset contains 3,448 unique sentences.

## Stratified train-test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df["sentence"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

In [ ]:
split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_test)],
        "negative_share": [
            (y_train == 0).mean(),
            (y_test == 0).mean(),
        ],
        "neutral_share": [
            (y_train == 1).mean(),
            (y_test == 1).mean(),
        ],
        "positive_share": [
            (y_train == 2).mean(),
            (y_test == 2).mean(),
        ],
    },
    index=["train", "test"],
).round(3)

split_summary

The 80/20 stratified split preserves class proportions across the training and held-out test sets. The test set remains untouched during feature and model development.